In [1]:
import os
import re
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# ============================================================
# КОНФИГУРАЦИЯ
# ============================================================

ROBERTA_NAME = "roberta-base"
MAX_LEN      = 128
SEED         = 42

emotion_classes = ["neutral", "happiness", "anger", "disgust", "sadness", "surprise", "fear"]

SPLITS_DIR = "splits"
os.makedirs(SPLITS_DIR, exist_ok=True)

# ============================================================
# ВОСПРОИЗВОДИМОСТЬ
# ============================================================

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# ============================================================
# ОЧИСТКА ТЕКСТА
# ============================================================

def clean_platform(text):
    text = str(text)
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'http[s]?://\S+', 'http', text)
    text = re.sub(r'www\.\S+', 'http', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def prepare_platform(df_orig):
    df = df_orig.copy()
    df['tweet_text'] = df['tweet_text'].apply(clean_platform)
    df = df[df['tweet_text'].str.strip().str.len() > 0]
    return df.reset_index(drop=True)

# ============================================================
# DATASET CLASS
# ============================================================

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

# ============================================================
# ПОДГОТОВКА И СОХРАНЕНИЕ СПЛИТОВ
# ============================================================

def prepare_and_save_splits(
    csv_path: str,
    splits_dir: str = SPLITS_DIR,
    test_size: float = 0.15,
    val_size: float  = 0.15,
    seed: int        = SEED,
):
    """
    Читает сырой CSV, чистит текст, кодирует метки,
    делает стратифицированный сплит train/val/test
    и сохраняет три CSV + маппинг меток в splits_dir.

    Файлы:
        splits_dir/train.csv
        splits_dir/val.csv
        splits_dir/test.csv
        splits_dir/label_mapping.csv   <- class_name <-> class_id
        splits_dir/split_info.txt      <- статистика
    """
    print(f"Reading: {csv_path}")
    df_raw   = pd.read_csv(csv_path)
    df_clean = prepare_platform(df_raw)

    # Фильтр по известным классам
    df = df_clean[df_clean["final_label"].isin(emotion_classes)].reset_index(drop=True)
    print(f"Samples after filtering: {len(df)}")

    # LabelEncoder
    le = LabelEncoder()
    le.fit(emotion_classes)
    df["label_id"] = le.transform(df["final_label"])

    # Распределение
    print("\nClass distribution:")
    dist = df["final_label"].value_counts()
    for cls in emotion_classes:
        n   = dist.get(cls, 0)
        pct = n / len(df) * 100
        print(f"  {cls:<12}: {n:>5}  ({pct:.1f}%)")

    # Сплит
    X_tv, X_test, y_tv, y_test = train_test_split(
        df["tweet_text"], df["label_id"],
        test_size=test_size, random_state=seed, stratify=df["label_id"],
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv,
        test_size=val_size, random_state=seed, stratify=y_tv,
    )

    print(f"\nSplit sizes  →  Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")

    # Сборка DataFrame-ов
    train_df = pd.DataFrame({"tweet_text": X_train, "label_id": y_train})
    val_df   = pd.DataFrame({"tweet_text": X_val,   "label_id": y_val})
    test_df  = pd.DataFrame({"tweet_text": X_test,  "label_id": y_test})

    # Добавляем человекочитаемую колонку
    id2cls = {i: cls for i, cls in enumerate(le.classes_)}
    for part_df in [train_df, val_df, test_df]:
        part_df["final_label"] = part_df["label_id"].map(id2cls)

    # Сохранение сплитов
    train_df.to_csv(os.path.join(splits_dir, "train.csv"), index=False)
    val_df.to_csv(os.path.join(splits_dir, "val.csv"),     index=False)
    test_df.to_csv(os.path.join(splits_dir, "test.csv"),   index=False)

    # Маппинг меток
    mapping_df = pd.DataFrame({
        "class_name": le.classes_,
        "class_id":   le.transform(le.classes_),
    })
    mapping_df.to_csv(os.path.join(splits_dir, "label_mapping.csv"), index=False)

    # Текстовая статистика
    info_lines = [
        f"Source file  : {csv_path}",
        f"Total samples: {len(df)}",
        f"Train size   : {len(train_df)}",
        f"Val size     : {len(val_df)}",
        f"Test size    : {len(test_df)}",
        f"Test frac    : {test_size}",
        f"Val frac     : {val_size}",
        f"Seed         : {seed}",
        "",
        "Class distribution (full dataset):",
    ]
    for cls in emotion_classes:
        n   = dist.get(cls, 0)
        pct = n / len(df) * 100
        info_lines.append(f"  {cls:<12}: {n:>5}  ({pct:.1f}%)")

    with open(os.path.join(splits_dir, "split_info.txt"), "w") as f:
        f.write("\n".join(info_lines))

    print(f"\nAll splits saved to: {splits_dir}/")
    print("  train.csv | val.csv | test.csv | label_mapping.csv | split_info.txt")

    return train_df, val_df, test_df, le


# ============================================================
# ЗАГРУЗКА ГОТОВЫХ СПЛИТОВ
# ============================================================
def load_splits(
    splits_dir: str = SPLITS_DIR,
    tokenizer_name: str = ROBERTA_NAME,
    max_len: int = MAX_LEN,
    batch_size: int = 32,
):
    """
    Загружает сохранённые сплиты и возвращает готовые DataLoader-ы.

    Возвращает
    ----------
    loaders : dict  {"train": DataLoader, "val": DataLoader, "test": DataLoader}
    meta    : dict  {"le": LabelEncoder, "num_labels": int,
                     "samples_per_class": list,
                     "X_train", "X_val", "X_test": pd.Series,
                     "y_train", "y_val", "y_test": pd.Series}
    """
    from torch.utils.data import DataLoader

    # Загрузка CSV
    train_df = pd.read_csv(os.path.join(splits_dir, "train.csv"))
    val_df   = pd.read_csv(os.path.join(splits_dir, "val.csv"))
    test_df  = pd.read_csv(os.path.join(splits_dir, "test.csv"))
    map_df   = pd.read_csv(os.path.join(splits_dir, "label_mapping.csv"))

    # Восстанавливаем LabelEncoder
    le = LabelEncoder()
    le.classes_ = np.array(map_df.sort_values("class_id")["class_name"].tolist())
    num_labels  = len(le.classes_)

    # Число примеров по классам в train (нужно для взвешенных лоссов)
    samples_per_class = [
        int((train_df["label_id"] == i).sum()) for i in range(num_labels)
    ]

    print(f"Splits loaded from: {splits_dir}/")
    print(f"  Train : {len(train_df)}")
    print(f"  Val   : {len(val_df)}")
    print(f"  Test  : {len(test_df)}")
    print(f"  Labels: {list(le.classes_)}")
    print(f"\n  Train samples per class:")
    for i, cls in enumerate(le.classes_):
        print(f"    {cls:<12}: {samples_per_class[i]:>5}")

    # Токенайзер
    print(f"\nLoading tokenizer: {tokenizer_name}")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    # Извлекаем X/y
    X_train = train_df["tweet_text"]
    y_train = train_df["label_id"]
    X_val   = val_df["tweet_text"]
    y_val   = val_df["label_id"]
    X_test  = test_df["tweet_text"]
    y_test  = test_df["label_id"]

    # Датасеты
    ds_train = TweetDataset(X_train, y_train, tokenizer, max_len)
    ds_val   = TweetDataset(X_val,   y_val,   tokenizer, max_len)
    ds_test  = TweetDataset(X_test,  y_test,  tokenizer, max_len)

    # DataLoader-ы
    loaders = {
        "train": DataLoader(ds_train, batch_size=batch_size,     shuffle=True),
        "val":   DataLoader(ds_val,   batch_size=batch_size * 2, shuffle=False),
        "test":  DataLoader(ds_test,  batch_size=batch_size * 2, shuffle=False),
    }

    meta = {
        "le":                le,
        "num_labels":        num_labels,
        "samples_per_class": samples_per_class,
        "X_train": X_train, "y_train": y_train,
        "X_val":   X_val,   "y_val":   y_val,
        "X_test":  X_test,  "y_test":  y_test,
    }

    return loaders, meta



# ============================================================
# ЗАПУСК ПОДГОТОВКИ
# ============================================================

train_df, val_df, test_df, le = prepare_and_save_splits(
    csv_path   = "full_annotated_dataset.csv",
    splits_dir = SPLITS_DIR,
)


/home/sh1rsh0v/emotions/jupyter_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading: full_annotated_dataset.csv
Samples after filtering: 9878

Class distribution:
  neutral     :  3205  (32.4%)
  happiness   :  2415  (24.4%)
  anger       :  1562  (15.8%)
  disgust     :   510  (5.2%)
  sadness     :  1464  (14.8%)
  surprise    :   509  (5.2%)
  fear        :   213  (2.2%)

Split sizes  →  Train: 7136  |  Val: 1260  |  Test: 1482

All splits saved to: splits/
  train.csv | val.csv | test.csv | label_mapping.csv | split_info.txt
